# 09 — Validações: NorthwindDW Spark SQL

14 verificações de integridade — todas via `spark.sql()`.

| # | Check | Esperado |
|---|-------|----------|
| 1-5 | Contagens e grains | 2155, 2155, True, 830, 0 |
| 6-7 | SCD2: 1 IsCurrent por chave natural | 0 violações |
| 8-11 | Integridade referencial FactSales → Dims | 0 órfãos |
| 12 | DimDate cobre OrderDateKeys | True |
| 13 | FactProductStock grain único | 0 duplicatas |
| 14 | NetRevenue ≤ GrossRevenue | 0 violações |

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR

spark = get_spark("NorthwindDW SQL - 09 Validation")
results = []

def check(num, desc, result, expected, check_fn=None):
    ok = check_fn(result, expected) if check_fn else (result == expected)
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] #{num}: {desc}\n       Resultado: {result} | Esperado: {expected}")
    results.append((num, desc, status, result, expected))
    return ok

def sql_count(q):
    return spark.sql(q).collect()[0][0]

print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 00:56:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 11}


In [3]:
n_od  = sql_count("SELECT COUNT(*) FROM bronze.order_details")
check(1, "bronze.order_details count", n_od, 2155)

n_fs = sql_count("SELECT COUNT(*) FROM gold.FactSales")
check(2, "gold.FactSales count", n_fs, 2155)
check(3, "FactSales == bronze.order_details", n_fs == n_od, True)

n_fof = sql_count("SELECT COUNT(*) FROM gold.FactOrderFulfillment")
check(4, "gold.FactOrderFulfillment count", n_fof, 830)

dups_fof = sql_count("""
    SELECT COUNT(*) FROM (
        SELECT OrderID FROM gold.FactOrderFulfillment GROUP BY OrderID HAVING COUNT(*) > 1
    )
""")
check(5, "FactOrderFulfillment grain único", dups_fof, 0)

[PASS] #1: bronze.order_details count
       Resultado: 2155 | Esperado: 2155


[PASS] #2: gold.FactSales count
       Resultado: 2155 | Esperado: 2155
[PASS] #3: FactSales == bronze.order_details
       Resultado: True | Esperado: True


[PASS] #4: gold.FactOrderFulfillment count
       Resultado: 830 | Esperado: 830


[PASS] #5: FactOrderFulfillment grain único
       Resultado: 0 | Esperado: 0


True

In [4]:
viol_cust = sql_count("""
    SELECT COUNT(*) FROM (
        SELECT CustomerID FROM gold.DimCustomer WHERE IsCurrent = true
        GROUP BY CustomerID HAVING COUNT(*) > 1
    )
""")
check(6, "DimCustomer: 1 IsCurrent por CustomerID", viol_cust, 0)

viol_prod = sql_count("""
    SELECT COUNT(*) FROM (
        SELECT ProductID FROM gold.DimProduct WHERE IsCurrent = true
        GROUP BY ProductID HAVING COUNT(*) > 1
    )
""")
check(7, "DimProduct: 1 IsCurrent por ProductID", viol_prod, 0)

[PASS] #6: DimCustomer: 1 IsCurrent por CustomerID
       Resultado: 0 | Esperado: 0


[PASS] #7: DimProduct: 1 IsCurrent por ProductID
       Resultado: 0 | Esperado: 0


True

In [5]:
for num, sk, dim in [
    (8,  "CustomerSK", "gold.DimCustomer"),
    (9,  "ProductSK",  "gold.DimProduct"),
    (10, "EmployeeSK", "gold.DimEmployee"),
    (11, "ShipperSK",  "gold.DimShipper"),
]:
    n = sql_count(f"SELECT COUNT(*) FROM gold.FactSales fs "
                  f"WHERE NOT EXISTS (SELECT 1 FROM {dim} d WHERE d.{sk} = fs.{sk})")
    check(num, f"FactSales: sem órfãos por {sk}", n, 0)

[PASS] #8: FactSales: sem órfãos por CustomerSK
       Resultado: 0 | Esperado: 0


[PASS] #9: FactSales: sem órfãos por ProductSK
       Resultado: 0 | Esperado: 0


[PASS] #10: FactSales: sem órfãos por EmployeeSK
       Resultado: 0 | Esperado: 0


[PASS] #11: FactSales: sem órfãos por ShipperSK
       Resultado: 0 | Esperado: 0


In [6]:
min_dk   = sql_count("SELECT MIN(OrderDateKey) FROM gold.FactSales")
max_dk   = sql_count("SELECT MAX(OrderDateKey) FROM gold.FactSales")
min_date = sql_count("SELECT MIN(DateKey) FROM gold.DimDate")
max_date = sql_count("SELECT MAX(DateKey) FROM gold.DimDate")
date_ok  = False if None in (min_dk, max_dk, min_date, max_date) else (min_date <= min_dk and max_date >= max_dk)
check(12, f"DimDate cobre OrderDateKeys ({min_dk}..{max_dk})", date_ok, True)

dups_stock = sql_count("""
    SELECT COUNT(*) FROM (
        SELECT SnapshotDateKey, ProductSK FROM gold.FactProductStock
        GROUP BY SnapshotDateKey, ProductSK HAVING COUNT(*) > 1
    )
""")
check(13, "FactProductStock grain único", dups_stock, 0)

viol_rev = sql_count("SELECT COUNT(*) FROM gold.FactSales WHERE NetRevenue > GrossRevenue + 0.01")
check(14, "NetRevenue <= GrossRevenue", viol_rev, 0)

[PASS] #12: DimDate cobre OrderDateKeys (19960704..19980506)
       Resultado: True | Esperado: True


[PASS] #13: FactProductStock grain único
       Resultado: 0 | Esperado: 0


[PASS] #14: NetRevenue <= GrossRevenue
       Resultado: 0 | Esperado: 0


True

In [7]:
passed = sum(1 for r in results if r[2] == "PASS")
failed = sum(1 for r in results if r[2] == "FAIL")
print("=" * 50)
print(f"Validações: {passed}/{len(results)} PASS | {failed} FAIL")
print("=" * 50)
if failed == 0:
    print("\nTodos os checks passaram! Pipeline Spark SQL validado.")
else:
    for r in results:
        if r[2] == "FAIL":
            print(f"  FAIL #{r[0]}: {r[1]} → {r[3]} (esperado: {r[4]})")

Validações: 14/14 PASS | 0 FAIL

Todos os checks passaram! Pipeline Spark SQL validado.
